# Aula 02 - Pontos cegos do retorno

Módulo 08 AdmTech, Computação · Semana 4 · Sprint 2 · Prof. José Romualdo

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/josercf/2026-2B-M08/blob/main/notebooks/aula02.ipynb)

A Parte A acompanha a aula: o exemplo de +5% e -2% do artigo do Turing Talks, o jogo da moeda com 10.000 jogadores e a diferença entre média aritmética e geométrica do capítulo 5 do Bodie, Kane e Marcus. A Parte B tem uma seção para cada grupo do comitê.

Roda no Colab pelo link acima ou no repositório clonado. Sem rede, lê o snapshot de `dados/`. Com `M08_OFFLINE=1`, a busca na internet fica desligada.

In [ ]:
import os

import numpy as np
import pandas as pd

DADOS_LOCAL = os.path.join("..", "dados")
DADOS_BRUTOS = "https://raw.githubusercontent.com/josercf/2026-2B-M08/main/dados/"


def ler_csv(nome, **kwargs):
    """Lê um CSV de dados/: do disco quando o repositório está clonado, da URL bruta no Colab."""
    local = os.path.join(DADOS_LOCAL, nome)
    if os.path.exists(local):
        return pd.read_csv(local, **kwargs)
    if os.environ.get("M08_OFFLINE") == "1":
        raise FileNotFoundError("%s não está em %s e M08_OFFLINE=1 desliga a rede" % (nome, DADOS_LOCAL))
    return pd.read_csv(DADOS_BRUTOS + nome, **kwargs)


datas = {"index_col": "data", "parse_dates": ["data"]}
precos = ler_csv("precos_ajustados.csv", **datas).ffill()  # Adj Close, com a regra de ausências da Aula 01
fechamento = ler_csv("precos_fechamento.csv", **datas).ffill()  # Close
cdi = ler_csv("cdi_diario.csv", **datas)["valor"]  # % ao dia
ipca = ler_csv("ipca_mensal.csv", **datas)["valor"]  # % ao mês
print("pregões:", len(precos), "de", precos.index[0].date(), "a", precos.index[-1].date())

## Parte A

### 1. Sobe 5%, cai 2%

O exemplo do autoestudo: uma ação sobe 5% num dia e cai 2% no dia seguinte. A soma dá 3%, só que a queda de 2% incide sobre um preço que já tinha subido. Retornos se compõem multiplicando `(1 + r)`.

In [ ]:
r = pd.Series([0.05, -0.02])
print("soma dos retornos:   %.2f%%" % (r.sum() * 100))
acumulado = (1 + r).cumprod() - 1
print("retorno do período:  %.2f%%" % (acumulado.iloc[-1] * 100))

### 2. O jogo da moeda com 10.000 jogadores

Cara rende +50%, coroa tira 40%, todo mundo começa com R$ 100. Antes de rodar a célula, anote o seu palpite para a mediana do saldo depois de 10 rodadas.

In [ ]:
rng = np.random.default_rng(2026)
jogadores, rodadas = 10_000, 10
caras = rng.random((jogadores, rodadas)) < 0.5
fatores = np.where(caras, 1.5, 0.6)
saldo = 100 * fatores.prod(axis=1)

print("retorno médio por rodada: %.2f%%" % ((fatores - 1).mean() * 100))
print("saldo médio:              R$ %.2f" % saldo.mean())
print("saldo mediano:            R$ %.2f" % np.median(saldo))
print("terminaram abaixo de R$ 100: %.1f%%" % ((saldo < 100).mean() * 100))

### 3. Média aritmética e média geométrica

A média aritmética por rodada é +5%. Quem joga vive a taxa composta, a média geométrica: `(1,5 × 0,6) ** (1/2) - 1`. O capítulo 5 do Bodie, Kane e Marcus aproxima a distância entre as duas por metade da variância.

In [ ]:
aritmetica = 0.5 * 0.50 + 0.5 * (-0.40)
geometrica = (1.5 * 0.6) ** 0.5 - 1
variancia = 0.5 * (0.50 - aritmetica) ** 2 + 0.5 * (-0.40 - aritmetica) ** 2

print("aritmética:                 %.2f%%" % (aritmetica * 100))
print("geométrica:                 %.2f%%" % (geometrica * 100))
print("aritmética - variância / 2: %.2f%%" % ((aritmetica - variancia / 2) * 100))

### 4. O mesmo efeito num ativo da cesta

Retorno do BBDC4 de fim de ano a fim de ano, de 2020 a 2025.

In [ ]:
fins = precos["BBDC4"].dropna().groupby(precos["BBDC4"].dropna().index.year).last()
anuais = fins.pct_change().loc[2020:2025]
print((anuais * 100).round(2).to_string())
print()
print("média aritmética: %.2f%% ao ano" % (anuais.mean() * 100))
print("média geométrica: %.2f%% ao ano" % (((1 + anuais).prod() ** (1 / len(anuais)) - 1) * 100))

## Parte B: a checagem do comitê

Cada grupo trabalha só na sua seção. A primeira célula do notebook já carregou `precos` (Adj Close tratado), `fechamento` (Close), `cdi` (% ao dia) e `ipca` (% ao mês). O número que decide o veredito vai para a ADR do grupo, no deck que o professor mandou no Slack.

<a id="g01"></a>
### G01 · BBAS3

A frase do assessor: "O BBAS3 deu prejuízo entre 2019 e 2026."

In [ ]:
serie = fechamento["BBAS3"].dropna()
# 1. A conta do assessor: retorno do primeiro ao último pregão.

# 2. A mesma conta com a outra visão de preço.

<a id="g02"></a>
### G02 · MGLU3

A frase do assessor: "A MGLU3 caiu 90% desde 2019. Uma alta de 90% devolve o cliente ao zero a zero."

In [ ]:
saldo = 100
# 1. Aplique a queda de 90% e, depois, a alta de 90%.

# 2. Alta que a MGLU3 precisa, a partir do último preço, para voltar ao primeiro pregão.

# 3. A regra: depois de uma queda x, qual alta devolve o ponto de partida?
def alta_necessaria(x):
    ...

<a id="g03"></a>
### G03 · PETR4

A frase do assessor: "A PETR4 tem volatilidade de 39% ao ano, então o pior que acontece em um ano é perder 39%."

In [ ]:
r = precos["PETR4"].pct_change(fill_method=None).dropna()
# 1. Volatilidade anual: desvio padrão diário vezes a raiz de 252.

# 2. Pior dia, pior janela de 252 pregões e queda do pico ao fundo.

<a id="g04"></a>
### G04 · WEGE3

A frase do assessor: "A WEGE3 ganhou da renda fixa no período."

In [ ]:
wege = precos["WEGE3"].dropna()
# 1. Retorno total da WEGE3 na janela.

# 2. Quanto R$ 100 viraram no CDI no mesmo período (cdi está em % ao dia).

# 3. Quanto a inflação levou (ipca está em % ao mês).